In [1]:
from pathlib import Path
import pandas as pd
from PyDI.io import load_parquet
from PyDI.entitymatching import EntityMatchingEvaluator
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 25)
pd.set_option('display.max_colwidth', 100)

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

PIPELINE_DIR = OUTPUT_DIR / "data_fusion"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

In [38]:
amazon_sample = load_parquet(
    DATA_DIR / "amazon_sample.parquet",
    name="amazon_sample"
)

goodreads_sample = load_parquet(
    DATA_DIR / "goodreads_sample.parquet",
    name="goodreads_sample"
)

metabooks_sample = load_parquet(
  DATA_DIR / "metabooks_sample.parquet",
  name="metabooks_sample"
)

import re

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r'<.*?>', '', t)
    # only change: space out dots between letters so initials stay separate
    t = re.sub(r'(?<=\b[a-z])\.\s*(?=[a-z]\b)', ' ', t)
    t = re.sub(r'[^a-z0-9\s]', '', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

def clean_author_field_goodreads(author_str: str) -> str:
    if not isinstance(author_str, str):
        return ""
    authors = []
    parts = [p.strip() for p in author_str.split(",") if p.strip()]
    for i, raw in enumerate(parts):
        has_paren = "(" in raw
        name = re.split(r"\s*\(", raw)[0].strip()  # drop parenthetical
        if len(name.split()) < 2:  # need first + surname
            continue
        if i == 0:
            authors.append(name)
            continue
        if has_paren:
            break  # stop at first later author with parentheses
        authors.append(name)
    return ", ".join(authors)

import re

def strip_all_parentheses(text: str) -> str:
    if text is None:
        return None

    s = str(text)

    # remove all ( ... ) groups, including multiple/nested, by repeated passes
    while "(" in s and ")" in s:
        new_s = re.sub(r"\([^()]*\)", "", s)
        if new_s == s:
            break
        s = new_s

    # collapse extra whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s

amazon_sample["title"]= amazon_sample["title"].apply(strip_all_parentheses)
metabooks_sample["title"]= metabooks_sample["title"].apply(strip_all_parentheses)
goodreads_sample["title"]= goodreads_sample["title"].apply(strip_all_parentheses)

amazon_sample['clean_title'] = amazon_sample['title'].apply(clean_text)
goodreads_sample['clean_title'] = goodreads_sample['title'].apply(clean_text)
metabooks_sample['clean_title'] = metabooks_sample['title'].apply(clean_text)
# Clean Author
amazon_sample["clean_author"] = amazon_sample["author"].apply(clean_text)
goodreads_sample["clean_author"] = goodreads_sample["author"].apply(clean_author_field_goodreads)
goodreads_sample["clean_author"] = goodreads_sample["clean_author"].apply(clean_text)
metabooks_sample["clean_author"] = metabooks_sample["author"].apply(clean_text)
#Clean Publisher
amazon_sample["clean_publisher"] = amazon_sample["publisher"].apply(clean_text)
metabooks_sample["clean_publisher"] = metabooks_sample["publisher"].apply(clean_text)
goodreads_sample["clean_publisher"] = goodreads_sample["publisher"].apply(clean_text)

In [40]:
from PyDI.io import load_csv

train_m2a = load_csv(
    MLDS_DIR / "train_MA.csv",
    name="train_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2a = load_csv(
    MLDS_DIR / "test_MA.csv",
    name="test_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

train_m2g = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="train_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2g = load_csv(
    MLDS_DIR / "test_MG.csv",
    name="test_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

In [41]:
from PyDI.entitymatching import EmbeddingBlocker

embedding_blocker_m2a = EmbeddingBlocker(
    metabooks_sample, amazon_sample,
    text_cols=['clean_title','clean_author','clean_publisher','publish_year'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=2,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)


embedding_blocker_m2g = EmbeddingBlocker(
    metabooks_sample, goodreads_sample,
    text_cols=['clean_title','clean_author','clean_publisher','publish_year'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=2,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

embedding_candidates_m2a = embedding_blocker_m2a.materialize()
embedding_candidates_m2g = embedding_blocker_m2g.materialize()

In [42]:
from PyDI.io import load_parquet
from PyDI.entitymatching import FeatureExtractor
from PyDI.entitymatching import StringComparator, NumericComparator


comparators = [
    # Title similarity
    StringComparator(column='clean_title',similarity_function='cosine'),
    
    # author similarity
    StringComparator(column='clean_author',similarity_function='cosine'),

    StringComparator(column='clean_publisher',similarity_function='cosine'),
    # publish year similarity
    NumericComparator(column='publish_year',method="absolute_difference",max_difference=2)
    ]

feature_extractor= FeatureExtractor(comparators)


train_features_m2a = feature_extractor.create_features(
    metabooks_sample, amazon_sample, train_m2a[['id1', 'id2']], labels=train_m2a['label'], id_column='id'
)

train_features_m2g = feature_extractor.create_features(
    metabooks_sample, goodreads_sample, train_m2g[['id1', 'id2']], labels=train_m2g['label'], id_column='id'
)

feature_columns_m2a = [col for col in train_features_m2a.columns if col not in ['id1', 'id2', 'label']]
X_train_m2a = train_features_m2a[feature_columns_m2a]
y_train_m2a = train_features_m2a['label']

feature_columns_m2g = [col for col in train_features_m2g.columns if col not in ['id1', 'id2', 'label']]
X_train_m2g = train_features_m2g[feature_columns_m2g]
y_train_m2g = train_features_m2g['label']

training_datasets = [(X_train_m2a, y_train_m2a),(X_train_m2g, y_train_m2g)]

In [43]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# classifiers
classifiers = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'SVC': SVC(probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42)
}

# parameter grids
param_grids = {
    'RandomForestClassifier': {
        'n_estimators': [50, 100, 200, 500],
        'max_depth': [None, 10, 20],
        'class_weight': ['balanced', None],
        'min_samples_split': [2, 5]
    },
    'GradientBoostingClassifier': {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5]
    },
    'SVC': {
        'C': [0.1, 1, 10],
        'kernel': ['linear'],
        'gamma': ['scale', 'auto']
    },
    'LogisticRegression': {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'liblinear']
    }
}

scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


best_models = []  # one best model per dataset

for (X_train, y_train) in training_datasets:
    grid_search_results = {}
    best_overall_score = -1
    best_overall_model = None
    best_model_name = None

    for name, model in classifiers.items():
        print(f"Running GridSearchCV for {name}...")
        
        grid = GridSearchCV(
            estimator=model,
            param_grid=param_grids[name],
            scoring=scorer,
            cv=cv_folds,
            n_jobs=-1,
            verbose=0
        )
        
        grid.fit(X_train, y_train)
        print(
            f"{name}: best F1 = {grid.best_score_:.4f} "
            f"with params {grid.best_params_}"
        )
        
        grid_search_results[name] = {
            'grid_search': grid,
            'best_score': grid.best_score_,
            'best_params': grid.best_params_,
            'best_estimator': grid.best_estimator_
        }

        if grid.best_score_ > best_overall_score:
            best_overall_score = grid.best_score_   
            best_overall_model = grid.best_estimator_
            best_model_name = name

    print(f"Best model for this dataset: {best_model_name} with F1={best_overall_score:.4f}")
    best_models.append(best_overall_model)

Running GridSearchCV for RandomForestClassifier...
RandomForestClassifier: best F1 = 0.9416 with params {'class_weight': None, 'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}
Running GridSearchCV for GradientBoostingClassifier...
GradientBoostingClassifier: best F1 = 0.9379 with params {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
Running GridSearchCV for SVC...
SVC: best F1 = 0.9216 with params {'C': 10, 'gamma': 'scale', 'kernel': 'linear'}
Running GridSearchCV for LogisticRegression...
LogisticRegression: best F1 = 0.9253 with params {'C': 1, 'penalty': 'l2', 'solver': 'lbfgs'}
Best model for this dataset: RandomForestClassifier with F1=0.9416
Running GridSearchCV for RandomForestClassifier...
RandomForestClassifier: best F1 = 0.8738 with params {'class_weight': None, 'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}
Running GridSearchCV for GradientBoostingClassifier...
GradientBoostingClassifier: best F1 = 0.8750 with params {'learning_r

In [75]:
best_models

[RandomForestClassifier(min_samples_split=5, random_state=42),
 SVC(C=10, kernel='linear', probability=True, random_state=42)]

In [77]:
# Feature Importances for the first model
importances=best_models[0].feature_importances_
feature_names = list(feature_columns_m2g)  

feat_imp = pd.Series(importances, index=feature_names)
feat_imp = feat_imp.sort_values(ascending=False)
display(feat_imp)    

StringComparator(clean_title, cosine, tokenization=word, list_strategy=None)        0.472308
StringComparator(clean_author, cosine, tokenization=word, list_strategy=None)       0.274705
StringComparator(clean_publisher, cosine, tokenization=word, list_strategy=None)    0.153026
NumericComparator(publish_year, absolute_difference, list_strategy=None)            0.099960
dtype: float64

In [76]:
pd.Series(best_models[1].coef_[0], index=feature_names).sort_values(ascending=False)

StringComparator(clean_title, cosine, tokenization=word, list_strategy=None)        2.605981
StringComparator(clean_publisher, cosine, tokenization=word, list_strategy=None)    1.647940
NumericComparator(publish_year, absolute_difference, list_strategy=None)            1.042175
StringComparator(clean_author, cosine, tokenization=word, list_strategy=None)       0.957886
dtype: float64

In [ ]:
from PyDI.entitymatching import StandardBlocker
from PyDI.entitymatching import MLBasedMatcher


ml_matcher = MLBasedMatcher(feature_extractor)

correspondences_m2a = ml_matcher.match(
    metabooks_sample, amazon_sample,
    candidates=embedding_candidates_m2a,
    use_probabilities=True,
    threshold=0.75,
    id_column='id',
    trained_classifier=best_models[0],
)

correspondences_m2g = ml_matcher.match(
    metabooks_sample, goodreads_sample,
    candidates=embedding_candidates_m2g,
    use_probabilities=True,
    threshold=0.65,
    id_column='id',
    trained_classifier=best_models[1]
)


In [45]:
display(EntityMatchingEvaluator.evaluate_matching(
        correspondences=correspondences_m2a,
        test_pairs=test_m2a,
        out_dir=BLOCK_EVAL_DIR,
    ))

display(EntityMatchingEvaluator.evaluate_matching(
        correspondences=correspondences_m2g,
        test_pairs=test_m2g,
        out_dir=BLOCK_EVAL_DIR,
    ))

{'precision': 0.9565217391304348,
 'recall': 0.927710843373494,
 'f1': 0.9418960244648319,
 'accuracy': 0.9218106995884774,
 'true_positives': 154,
 'false_positives': 7,
 'false_negatives': 12,
 'true_negatives': 70,
 'threshold_used': 0.0,
 'total_correspondences': 28304,
 'filtered_correspondences': 28304,
 'evaluation_timestamp': '2025-12-20T02:27:51.619443',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_detailed_results.csv']}

{'precision': 0.9538461538461539,
 'recall': 0.8611111111111112,
 'f1': 0.9051094890510949,
 'accuracy': 0.891213389121339,
 'true_positives': 124,
 'false_positives': 6,
 'false_negatives': 20,
 'true_negatives': 89,
 'threshold_used': 0.0,
 'total_correspondences': 6048,
 'filtered_correspondences': 6048,
 'evaluation_timestamp': '2025-12-20T02:27:52.345339',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_detailed_results.csv']}

In [54]:
cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)
cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2a,
    out_dir=cluster_analysis_dir
)
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2g,
    out_dir=cluster_analysis_dir
)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)


📊 Cluster Size Distribution Results (Metabooks -> Amazon):


,cluster_size,frequency,percentage
0,2,21313,90.956811
1,3,784,3.345852
2,4,1067,4.553602
3,5,109,0.465176
4,6,86,0.367019
5,7,18,0.076818
6,8,28,0.119495
7,9,10,0.042677
8,10,5,0.021338
9,11,3,0.012803



📊 Cluster Size Distribution Results (Metabooks -> Goodreads):


,cluster_size,frequency,percentage
0,2,5055,92.446964
1,3,329,6.016825
2,4,54,0.987564
3,5,16,0.292612
4,6,3,0.054865
5,7,6,0.109729
6,8,2,0.036576
7,9,2,0.036576
8,11,1,0.018288


In [55]:
from PyDI.entitymatching import MaximumBipartiteMatching

# We are using Maxmimum Bipartite Matching to refine results to 1:1 matches
clusterer = MaximumBipartiteMatching()
mbm_correspondences_m2a = clusterer.cluster(correspondences_m2a)
mbm_correspondences_m2g = clusterer.cluster(correspondences_m2g)
all_correspondences = pd.concat([mbm_correspondences_m2a, mbm_correspondences_m2g], ignore_index=True)
len(all_correspondences)

30533

In [56]:
display(EntityMatchingEvaluator.evaluate_matching(
        correspondences=mbm_correspondences_m2a,
        test_pairs=test_m2a,
        out_dir=BLOCK_EVAL_DIR,
    ))

display(EntityMatchingEvaluator.evaluate_matching(
        correspondences=mbm_correspondences_m2g,
        test_pairs=test_m2g,
        out_dir=BLOCK_EVAL_DIR,
    ))

{'precision': 0.987012987012987,
 'recall': 0.9156626506024096,
 'f1': 0.95,
 'accuracy': 0.934156378600823,
 'true_positives': 152,
 'false_positives': 2,
 'false_negatives': 14,
 'true_negatives': 75,
 'threshold_used': 0.0,
 'total_correspondences': 24983,
 'filtered_correspondences': 24983,
 'evaluation_timestamp': '2025-12-20T02:29:11.353186',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_detailed_results.csv']}

{'precision': 0.967741935483871,
 'recall': 0.8333333333333334,
 'f1': 0.8955223880597015,
 'accuracy': 0.8828451882845189,
 'true_positives': 120,
 'false_positives': 4,
 'false_negatives': 24,
 'true_negatives': 91,
 'threshold_used': 0.0,
 'total_correspondences': 5550,
 'filtered_correspondences': 5550,
 'evaluation_timestamp': '2025-12-20T02:29:11.920633',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_detailed_results.csv']}

In [57]:
from PyDI.fusion import DataFusionStrategy, DataFusionEngine, union, prefer_higher_trust,voting
import pandas as pd

metabooks_sample.attrs["trust_score"] = 3
goodreads_sample.attrs["trust_score"] = 2
amazon_sample.attrs["trust_score"] = 1
strategy = DataFusionStrategy('books_fusion_strategy')

strategy.add_attribute_fuser('title', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('author', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publish_year', voting)
strategy.add_attribute_fuser('publisher', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('language', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('page_count', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('genres',union)

engine = DataFusionEngine(strategy, debug=True, debug_format='json',
                          debug_file= PIPELINE_DIR / "debug_fusion_ml_embedding_blocker.jsonl")

fused_ml_emblocker = engine.run(
    datasets=[amazon_sample, metabooks_sample, goodreads_sample],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False
)
fused_ml_emblocker.to_parquet(PIPELINE_DIR / "fused_ml_emblocker.parquet")
print(f'Fused rows: {len(fused_ml_emblocker):,}')

Fused rows: 28,732


In [58]:
from PyDI.fusion import tokenized_match, boolean_match,numeric_tolerance_match,set_equality_match,exact_match

import numpy as np
import re, ast, numpy as np, pandas as pd


def categories_set_equal(a, b) -> bool:
    """Return True if a and b contain the same unique categories (order/type agnostic)."""
    def to_set(x):
        def items(v):
            # missing
            if v is None or (isinstance(v, float) and np.isnan(v)): return []
            # numpy array → recurse over elements
            if isinstance(v, np.ndarray): 
                out=[]; [out.extend(items(e)) for e in v.flatten()]; return out
            # python containers → recurse over elements
            if isinstance(v, (list, tuple, set)):
                out=[]; [out.extend(items(e)) for e in v]; return out
            # scalar/string: try parse stringified list; else split by delimiters
            s = str(v).strip()
            if s == "" or s.lower() in {"nan","none"}: return []
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)): return items(parsed)
            except Exception:
                pass
            return [p.strip() for p in re.split(r"[|,;/]", s) if p.strip()]
        return {it.lower() for it in items(x)}
    return to_set(a) == to_set(b)

strategy.add_evaluation_function("title", exact_match)
strategy.add_evaluation_function("author", tokenized_match)
strategy.add_evaluation_function("publisher", tokenized_match)
strategy.add_evaluation_function("publish_year", numeric_tolerance_match)
strategy.add_evaluation_function("page_count", numeric_tolerance_match)
strategy.add_evaluation_function("language", tokenized_match)
strategy.add_evaluation_function("genres", categories_set_equal)

fused_dataset = pd.read_parquet(PIPELINE_DIR / "fused_ml_emblocker.parquet")
golden_fused_dataset= pd.read_csv(MLDS_DIR / "golden_fused_books.csv")

In [59]:
from PyDI.fusion import DataFusionEvaluator
fused_dataset.drop_duplicates(subset='isbn_clean', keep='first',inplace=True)
# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused_dataset,
    fused_id_column='isbn_clean',
    gold_df=golden_fused_dataset,
    gold_id_column='isbn_clean',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

Evaluating fusion results against gold standard...

Fusion Evaluation Results:
  overall_accuracy: 0.829
  macro_accuracy: 0.829
  num_evaluated_records: 20
  num_evaluated_attributes: 7
  total_evaluations: 140
  total_correct: 116
  genres_accuracy: 0.850
  genres_count: 20
  title_accuracy: 0.600
  title_count: 20
  language_accuracy: 0.900
  language_count: 20
  publish_year_accuracy: 0.900
  publish_year_count: 20
  page_count_accuracy: 0.850
  page_count_count: 20
  publisher_accuracy: 0.850
  publisher_count: 20
  author_accuracy: 0.850
  author_count: 20

Overall Accuracy: 82.9%
